# Analysis of xanf pipeline run
Updated organelle segmentation. Previously, I was removing any masks that touched the ROI boundary. Unfortunately, this was causing segmentation issues for antibody panel 2. Thus, I have updated the approach so the ROI boundary will simply clip masks that cross the boundary. This should retain the structure as best as possible. 

In [1]:
import pathlib as Path
import pandas as pd
import os

import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load metadata

In [2]:
METADATA_FN = "/data/kelpschdj/iNDI/Production/notebooks/analysis_xanf/COMBINED_ASP_BATCH_METADATA.csv"
METADATA = pd.read_csv(METADATA_FN)

print(METADATA.head())

                                        Cell_Line_ID iNDI_Plate_ID Well_ID  \
0                 JAX_ID_JIPSC001002\nFUS R495X HOM     INDI00004D     A01   
1  JAX_ID_JIPSC001072\nVCP R159H REV \nLow counts...    INDI00004D     A02   
2              JAX_ID_JIPSC001000\nParental  ID:5424    INDI00004D     A03   
3                 JAX_ID_JIPSC001006\nFUS R495X REV     INDI00004D     A04   
4                  JAX_ID_JIPSC001070\nVCP R159H HET    INDI00004D     A05   

   Genotype_Name       JAX_ID Batch_Info      Gene Variant Genotype  \
0  FUS R495X HOM  JIPSC001002        NaN       FUS   R495X  SNV/SNV   
1  VCP R159H REV  JIPSC001072        NaN       VCP   R159H   REV/WT   
2       Parental  JIPSC001000    ID:5424  KOLF2.1J     NaN      NaN   
3  FUS R495X REV  JIPSC001006        NaN       FUS   R495X   REV/WT   
4  VCP R159H HET  JIPSC001070        NaN       VCP   R159H   SNV/WT   

      JAX_ID.1 Asp PosID  AbPanel  Batch     Layout  \
0  JIPSC001002        A2        2      1  Plate00

# Organelle feature data
Pull in organelle feature data (Lysosome, Endosome, Mitochondria)

In [3]:
files = glob.glob("/data/kelpschdj/iNDI/Production/outputs/run_20261116_165635_xanf/organelle_features/*_organelle_features_*.parquet")

parts = []
for f in files:
    part = pd.read_parquet(f)
    parts.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts, ignore_index=True)

print(f"\nLoaded {len(files)} files, {len(df)} total rows")
print(df.head())
print(df.columns.tolist())

0acd04e9-3bc9-400b-8557-3f5b4123caaa_organelle_features__20261117_123158.parquet: 236500 rows
106c8b69-e11f-4698-abd6-d4d01ec5fbb2_organelle_features__20261117_123158.parquet: 113652 rows
48fb9a14-1123-488d-9b56-3befe944f933_organelle_features__20261117_123158.parquet: 313827 rows
72a1d9f3-277a-4abd-bf55-cbbc07152dff_organelle_features__20261117_123158.parquet: 61168 rows
84809677-0531-486c-b79f-e25c7ef9a7c6_organelle_features__20261117_123158.parquet: 201110 rows
85cc6732-9e29-40ad-9ec1-e94b9f334ebe_organelle_features__20261117_123158.parquet: 139291 rows
9ed6a73a-227d-45a5-90dd-5c87fa906bea_organelle_features__20261117_123158.parquet: 256034 rows
a9b2a502-ad2f-4ee9-85ba-10df3cea3b46_organelle_features__20261117_123158.parquet: 378626 rows
cba96cb5-1337-4136-ab55-482e1bc9bed3_organelle_features__20261117_123158.parquet: 296547 rows
cd5eb985-d862-4c96-8b57-570f462ee2fa_organelle_features__20261117_123158.parquet: 358116 rows
d5847ccd-b4ba-4ce9-a214-ea68b817e7f0_organelle_features__2026

## Merge with metadata 

In [4]:
print(df.columns.tolist())
print(METADATA.columns.tolist())

print(METADATA["Well_ID"].head())
print(df[["Row", "Column"]].head())

df["Well_ID"] = (
    df["Row"].apply(lambda r: chr(ord("A") + int(r) - 1))
    + df["Column"].astype(int).astype(str).str.zfill(2)
)

df = df.merge(
    METADATA,
    left_on=["Measurement_ID", "Well_ID"],
    right_on=["Folder_ID", "Well_ID"],
    how="left",
)

['Measurement_ID', 'subdirectory', 'Row', 'Column', 'Frame', 'Plane', 'Time', 'DAPI_filename', 'channel_filename', 'Structure', 'Stain', 'Nucleus_ID', 'frame_std', 'contrast_check', 'area_px', 'organelle_count', 'average_organelle_area', 'roi_occupancy', 'n_truncated_objects', 'truncated_area_fraction', 'max_f_intensity', 'sum_f_intensity', 'mean_f_intensity', 'median_f_intensity', 'CoefOfVar_intensity', 'area_mean', 'area_median', 'area_std', 'area_p10', 'area_p90', 'perimeter_mean', 'perimeter_median', 'perimeter_std', 'perimeter_p10', 'perimeter_p90', 'eccentricity_mean', 'eccentricity_median', 'eccentricity_std', 'eccentricity_p10', 'eccentricity_p90', 'solidity_mean', 'solidity_median', 'solidity_std', 'solidity_p10', 'solidity_p90', 'extent_mean', 'extent_median', 'extent_std', 'extent_p10', 'extent_p90', 'axis_major_length_mean', 'axis_major_length_median', 'axis_major_length_std', 'axis_major_length_p10', 'axis_major_length_p90', 'axis_minor_length_mean', 'axis_minor_length_med

## Well-level summary

In [8]:
for plate, grp in df.groupby('iNDI_Plate_ID'):
    print(f"\n{plate}")
    print(grp.groupby(['Gene', 'Variant', 'Genotype'])['Well_ID'].nunique())


INDI00004D
Gene  Variant  Genotype
FUS   R495X    REV/WT      48
               SNV/SNV     48
               SNV/WT      48
VCP   R159H    REV/WT      48
               SNV/SNV     48
               SNV/WT      47
Name: Well_ID, dtype: int64

INDI00008D
Gene    Variant  Genotype
TARDBP  M337V    REV/WT      48
                 SNV/WT      48
        Q331K    SNV/SNV     48
TIA1    P351L    REV/WT      48
                 SNV/SNV     48
Name: Well_ID, dtype: int64

INDI00016D
Gene   Variant  Genotype
APP    A673T    REV/WT      48
                SNV/WT      48
       E693G    SNV/SNV     48
                SNV/WT      48
       V717I    SNV/SNV     48
PSEN1  E276A    SNV/WT      48
Name: Well_ID, dtype: int64

INDI00020D
Gene   Variant  Genotype
APP    E693G    REV/WT      20
FUS    R216C    REV/WT      48
                SNV/WT      48
PSEN1  E276A    REV/WT      47
Name: Well_ID, dtype: int64

INDI00024D
Gene  Variant  Genotype
MAPT  R406W    REV/WT      48
               SNV/SNV  

## Clean-up data
If a nuclear ROI does not have segmentation for all three channels, throw it away

In [9]:
## add new analysis to check number of channels per nucleus is 4 (including nuclear channel) for all nuclei in the dataset - If a nucleus has less than 4 channels remove it from analysis
counts = (
    df.groupby(["DAPI_filename", "Nucleus_ID"])["Structure"]
        .nunique()
)

keep_keys = counts[counts == 3].index


filtered = df.set_index(["DAPI_filename", "Nucleus_ID"]).loc[keep_keys].reset_index()
print(f"Filtered dataset: {len(filtered)} rows (from {len(df)} original rows)")

Filtered dataset: 3081229 rows (from 3112986 original rows)


Reduced by ~ 31000 ROIs

In [10]:
df = filtered  # replace the original df with the filtered one for further analysis

## Apply subjective filters from Marianita
Alena applied this to the well level data - not the ROI level. Would that resolve to the same effect????\
Alena has 1140 parental automated wells

In [ ]:
# # MS_filter 
# MS_filter_fn = "/data/CARDPB2/iNDI/Production/metadata/AbPanel1/marianita_excluded_genotypes_by_plate_upd_10092026.csv"
# MS_filter = pd.read_csv(MS_filter_fn)

# MS_filter["JAX_Code_Marianita"] = (
#     MS_filter["JAX_Code_Marianita"].astype(str).str.removeprefix("JAX_ID_")
# )
# MS_filter = MS_filter.rename(columns={"JAX_Code_Marianita": "JAX_ID"})

# print("MS_filter columns:", MS_filter.columns.tolist())
# print(MS_filter.head())

# remove_keys = set(zip(
#     MS_filter["iNDI_Plate_ID"].astype(str).str.strip(),
#     MS_filter["Asp PosID"].astype(str).str.strip(),
#     MS_filter["JAX_ID"].astype(str).str.strip(),
# ))

# key = list(zip(
#     df["iNDI_Plate_ID"].astype(str).str.strip(),
#     df["Asp PosID"].astype(str).str.strip(),
#     df["JAX_ID"].astype(str).str.strip(),
# ))
# mask = pd.Series(key, index=df.index).isin(remove_keys)
# filtered = df[~mask].copy()
# print(f"removed {mask.sum()} rows; {len(filtered)} remain")

In [13]:
print(filtered.head())
print(filtered.shape)
# pull unique DAPI_filename, Measurment_ID, Nucleus_ID

filter_key = filtered[["DAPI_filename", "Measurement_ID", "Nucleus_ID", "JAX_ID"]].drop_duplicates().reset_index(drop=True)
filter_key = filter_key.rename(columns = {"DAPI_filename": "filename", "Nucleus_ID": "label"})
print(filter_key.head())
print(filter_key.shape)

               DAPI_filename  Nucleus_ID  \
0  r01c01f01p01-ch01t01.tiff         1.0   
1  r01c01f01p01-ch01t01.tiff         1.0   
2  r01c01f01p01-ch01t01.tiff         1.0   
3  r01c01f01p01-ch01t01.tiff         2.0   
4  r01c01f01p01-ch01t01.tiff         2.0   

                         Measurement_ID subdirectory  Row  Column  Frame  \
0  d5b2e3eb-3b91-4619-8c74-846bbcd92bd4       r01c01    1       1      1   
1  d5b2e3eb-3b91-4619-8c74-846bbcd92bd4       r01c01    1       1      1   
2  d5b2e3eb-3b91-4619-8c74-846bbcd92bd4       r01c01    1       1      1   
3  a9b2a502-ad2f-4ee9-85ba-10df3cea3b46       r01c01    1       1      1   
4  a9b2a502-ad2f-4ee9-85ba-10df3cea3b46       r01c01    1       1      1   

   Plane  Time           channel_filename  ... Batch_Info  Gene  Variant  \
0      1     1  r01c01f01p01-ch04t01.tiff  ...        NaN  TIA1    P351L   
1      1     1  r01c01f01p01-ch02t01.tiff  ...        NaN  TIA1    P351L   
2      1     1  r01c01f01p01-ch03t01.tiff  ...    

## Write final output

In [14]:
filtered.to_parquet("xanf_organelle_output.parquet", index=False)

# Nucleus feature data

## Merge with metadata

In [15]:
files = glob.glob("/data/kelpschdj/iNDI/Production/outputs/run_20261116_165635_xanf/nuclei_filtered/*_nuclei_filtered.parquet")

parts = []
for f in files:
    part = pd.read_parquet(f)
    parts.append(part)
    print(f"{os.path.basename(f)}: {len(part)} rows")

df = pd.concat(parts, ignore_index=True)

print(f"\nLoaded {len(files)} files, {len(df)} total rows")
print(df.head())
print(df.columns.tolist())
print(df.shape)

df["Well_ID"] = (
    df["Row"].apply(lambda r: chr(ord("A") + int(r) - 1))
    + df["Column"].astype(int).astype(str).str.zfill(2)
)

df = df.merge(
    METADATA,
    left_on=["Measurement_ID", "Well_ID"],
    right_on=["Folder_ID", "Well_ID"],
    how="left",
)

print(df.head())
print(df.shape)

0acd04e9-3bc9-400b-8557-3f5b4123caaa_nuclei_filtered.parquet: 162722 rows
106c8b69-e11f-4698-abd6-d4d01ec5fbb2_nuclei_filtered.parquet: 82775 rows
48fb9a14-1123-488d-9b56-3befe944f933_nuclei_filtered.parquet: 224594 rows
72a1d9f3-277a-4abd-bf55-cbbc07152dff_nuclei_filtered.parquet: 68159 rows
84809677-0531-486c-b79f-e25c7ef9a7c6_nuclei_filtered.parquet: 133722 rows
85cc6732-9e29-40ad-9ec1-e94b9f334ebe_nuclei_filtered.parquet: 87508 rows
9ed6a73a-227d-45a5-90dd-5c87fa906bea_nuclei_filtered.parquet: 171742 rows
a9b2a502-ad2f-4ee9-85ba-10df3cea3b46_nuclei_filtered.parquet: 237108 rows
cba96cb5-1337-4136-ab55-482e1bc9bed3_nuclei_filtered.parquet: 182607 rows
cd5eb985-d862-4c96-8b57-570f462ee2fa_nuclei_filtered.parquet: 268737 rows
d5847ccd-b4ba-4ce9-a214-ea68b817e7f0_nuclei_filtered.parquet: 129787 rows
d5b2e3eb-3b91-4619-8c74-846bbcd92bd4_nuclei_filtered.parquet: 190444 rows
e61d5e8c-faef-4c2c-8df6-6cc72032f19e_nuclei_filtered.parquet: 176028 rows

Loaded 13 files, 2115933 total rows
   l

## QC filter

In [16]:
keys = ["filename", "Measurement_ID", "label", "JAX_ID"]
subset = df[df.set_index(keys).index.isin(filter_key.set_index(keys).index)]
print(subset.shape)

(1054301, 56)


## Write final output

In [17]:
subset.to_parquet("xanf_nucleus_output.parquet", index=False)